#### PARTIE 1 : Définir une classe de contexte structurée appelée ColourContext

In [17]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

#### PARTIE 2 : Agent sans contexte

In [18]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

model = ChatOllama(
    model="llama3.2:3b", # ou mistral, gemma, etc.
)

agent = create_agent(
    model=model,
    context_schema=ColourContext
)

In [19]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

print(response['messages'][-1].content)

I don't have any information about your personal preferences, including your favorite color. This conversation just started, and I'm here to help with any questions or topics you'd like to discuss. Would you like to share your favorite color with me?


#### PARTIE 3 : Agent avec contexte

In [20]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

In [21]:
@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [22]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [23]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

print(response['messages'][-1].content)

c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


The answer to your original question, 'What is your favourite colour?', is blue.


#### PARTIE 4 : Changement de contexte

In [24]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

print(response['messages'][-1].content)

c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


The user's favourite colour is green.


#### PARTIE 5 : Définir un état personnalisé d’agent en héritant de AgentState

In [25]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

#### PARTIE 6 : Agent qui modifie un état

In [26]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
    })

In [27]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [28]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

Your favourite colour has been successfully updated to green! Would you like me to do anything else for you?


#### PARTIE 7 : Agent qui récupère un état

In [29]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

In [30]:
agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [31]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

I've taken note that your favourite colour is indeed green. Would you like to ask another question or do something else?


In [32]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

I've retrieved your favourite colour, which is green. Is there anything else I can help you with?
